# Data Comparison
The objective of this stage is to perform a systematic reconciliation between the OMS and WMS datasets to identify all discrepancies and quantify their impact. Having standardized the data in the Validation phase, now we perform a direct "side-by-side" analysis to pinpoint exactly where the two systems diverge.

This step includes:
- Multi-level Matching: Using composite reconciliation key (`InvoiceNo` + `StockCode` + `Quantity`) to align records across both datasets at the most granular level.

- Completeness Gap Analysis: Full Outer Join to identify orphan records:
    - OMS-only records: transactions in OMS but missing in WMS (potential lost shipments)
    - WMS-only records: transactions in WMS but missing in OMS (potential ghost records)

- Numerical Variance Calculation: Quantifying differences in UnitPrice and other metrics for matched records.

- Duplicate & Error Impact: Analyzing how flagged records (duplicates, invalid dates, missing prices) affect reconciliation results.

- Financial Impact Assessment: Aggregating variances into total Net Variance showing financial exposure.

- Root Cause Cross-Referencing: Linking discrepancies back to Validation flags to identify systematic issues.


**Step 1** Data load

In [ ]:
import pandas as pd
import ast

def load_data(file_path):
    try:
        df = pd.read_csv(file_path, encoding='latin1')
        print(f"File {file_path} loaded correctly using latin1 encoding. Loaded {len(df)} rows.")
    except Exception:
        df = pd.read_csv(file_path, encoding='cp1252')
        print(f"File {file_path} loaded correctly using cp1252 encoding. Loaded {len(df)} rows.")
    
    df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors='coerce')
    
    df['validation_flags'] = df['validation_flags'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else [])
    
    return df

df_oms = load_data('validation/oms_final_clean.csv')
df_wms = load_data('validation/wms_final_clean.csv')

#flags in wms
wms_with_flags = df_wms[df_wms['validation_flags'].apply(lambda x: len(x) > 0)]
print(f"WMS records with flags: {len(wms_with_flags)}")
print(wms_with_flags[['InvoiceNo', 'StockCode', 'Quantity', 'validation_flags']].head(15))

# flags in oms
oms_with_flags = df_oms[df_oms['validation_flags'].apply(lambda x: len(x) > 0)]
print(f"\nOMS records with flags: {len(oms_with_flags)}")
print(oms_with_flags[['InvoiceNo', 'StockCode', 'Quantity', 'validation_flags']].head(15))


File validation/oms_final_clean.csv loaded correctly using latin1 encoding. Loaded 541909 rows.
File validation/wms_final_clean.csv loaded correctly using latin1 encoding. Loaded 542409 rows.
WMS records with flags: 12093
    InvoiceNo StockCode  Quantity     validation_flags
0      536365    85123A         6       [Invalid_Date]
1      536365     71053         6       [Invalid_Date]
2      536365    84406B         8       [Invalid_Date]
3      536365    84029G         6       [Invalid_Date]
4      536365    84029E         6       [Invalid_Date]
5      536365     22752         2       [Invalid_Date]
6      536365     21730         6       [Invalid_Date]
7      536366     22633         6       [Invalid_Date]
8      536366     22632         6       [Invalid_Date]
9      536367     84879        32       [Invalid_Date]
10     536367     22745         6       [Invalid_Date]
295    536396     21730         6  [Missing_UnitPrice]
485    536409     22111         1   [Duplicate_Record]
489    5

Observations:
- WMS contains 12,093 flagged records (2.23% of total), primarily consisting of duplicate records with some data quality issues (Invalid_Date, Missing_UnitPrice)
- OMS contains 10,147 flagged records (1.87% of total), exclusively duplicate records
- Invalid_Date flags in WMS (11 records) correspond to the intentional data corruption introduced at the beginning of the analysis
- The presence of flagged records in both datasets indicates that the Validation phase successfully identified data quality issues without removing them, allowing for impact analysis
- Duplicate records represent the largest category of flags in both systems, suggesting potential data entry errors or system synchronization issues

**Step 2** Multilevel matching  - data preparation for matching

In [56]:
# key for reconciliation
df_oms['reconciliation_key'] = df_oms['InvoiceNo'].astype(str) + '_' + df_oms['StockCode'].astype(str) + '_' + df_oms['Quantity'].astype(str)
df_wms['reconciliation_key'] = df_wms['InvoiceNo'].astype(str) + '_' + df_wms['StockCode'].astype(str) + '_' + df_wms['Quantity'].astype(str)

# Merge based on key
df_matched = pd.merge(df_oms, df_wms, on='reconciliation_key', how='inner', suffixes=('_oms', '_wms'))
print(f"Matched records: {len(df_matched)}")


Matched records: 555007


**Note**: 
A full outer join on the raw data showed an unnatural increase in the number of records (555,007 for databases with approximately 542,000). Analysis revealed the presence of many-to-many relationships (e.g., invoice 555524 generated 400 rows after the join). This is the result of mass duplication of identical order lines in the source systems.

In [59]:
# Checking the records that generated the most rows in the join
dupe_check = df_matched.groupby(['InvoiceNo_oms', 'StockCode_oms', 'Quantity_oms']).size().reset_index(name='row_count')
print(dupe_check.sort_values(by='row_count', ascending=False).head(10))

       InvoiceNo_oms StockCode_oms  Quantity_oms  row_count
207531        555524         22698             1        400
529092       C544580             S            -1        256
207530        555524         22697             1        144
408575        572861         22775            12         64
530714       C553531             S            -1         49
529094       C544583             S            -1         49
481701        578289         23395             1         36
25683         538514         21756             1         36
403039        572344             M            48         36
531661       C558347             S            -1         36


In [ ]:
print("\n AGGREGATING DUPLICATES")

# calculate duplicates before aggregation
oms_duplicates_count = len(df_oms) - df_oms[['InvoiceNo', 'StockCode', 'Quantity']].drop_duplicates().shape[0]
wms_duplicates_count = len(df_wms) - df_wms[['InvoiceNo', 'StockCode', 'Quantity']].drop_duplicates().shape[0]

print(f"OMS duplicates before aggregation: {oms_duplicates_count}")
print(f"WMS duplicates before aggregation: {wms_duplicates_count}")

# aggregation
df_oms_agg = df_oms.groupby(['InvoiceNo', 'StockCode', 'Quantity']).agg({
    'UnitPrice': 'first',
    'InvoiceDate': 'first',
    'CustomerID': 'first',
    'Country': 'first',
    'Description': 'first',
    'validation_status': 'first',
    'validation_flags': 'first'
}).reset_index()

df_wms_agg = df_wms.groupby(['InvoiceNo', 'StockCode', 'Quantity']).agg({
    'UnitPrice': 'first',
    'InvoiceDate': 'first',
    'CustomerID': 'first',
    'Country': 'first',
    'Description': 'first',
    'validation_status': 'first',
    'validation_flags': 'first'
}).reset_index()

print(f"\nOMS after aggregation: {len(df_oms_agg)} unique records")
print(f"WMS after aggregation: {len(df_wms_agg)} unique records")


 AGGREGATING DUPLICATES
OMS duplicates before aggregation: 5431
WMS duplicates before aggregation: 5931

OMS after aggregation: 536478 unique records
WMS after aggregation: 536478 unique records


In [61]:
df_matched = pd.merge(df_oms_agg, df_wms_agg, 
                      on=['InvoiceNo', 'StockCode', 'Quantity'],
                      how='inner', suffixes=('_oms', '_wms'))

print(f"Matched records: {len(df_matched)}")

Matched records: 536478


**Step 3** Completeness gap analysis

In [62]:
print("\nCOMPLETENESS GAP ANALYSIS")

df_full_outer = pd.merge(df_oms_agg[['InvoiceNo', 'StockCode', 'Quantity']], 
                          df_wms_agg[['InvoiceNo', 'StockCode', 'Quantity']], 
                          on=['InvoiceNo', 'StockCode', 'Quantity'],
                          how='outer', indicator=True)

oms_only = df_full_outer[df_full_outer['_merge'] == 'left_only']
wms_only = df_full_outer[df_full_outer['_merge'] == 'right_only']

print(f"OMS-only records (missing in WMS): {len(oms_only)}")
print(f"WMS-only records (missing in OMS): {len(wms_only)}")


COMPLETENESS GAP ANALYSIS
OMS-only records (missing in WMS): 0
WMS-only records (missing in OMS): 0


Observations:
- Both OMS-only (0) and WMS-only (0) record counts indicate complete bidirectional coverage. Every unique transaction (defined by InvoiceNo + StockCode + Quantity combination) exists in both systems.
- The absence of missing records suggests that the core transaction data is properly synchronized between the two systems. No lost shipments or ghost records are present at the aggregated level. 

**Step 4** Numerical Variance*

In [63]:
print("\nNUMERICAL VARIANCE")

df_matched['UnitPrice_variance'] = df_matched['UnitPrice_wms'] - df_matched['UnitPrice_oms']
df_matched['UnitPrice_variance_pct'] = (df_matched['UnitPrice_variance'] / df_matched['UnitPrice_oms'] * 100).round(2)

df_matched['Total_Value_oms'] = df_matched['Quantity'] * df_matched['UnitPrice_oms']
df_matched['Total_Value_wms'] = df_matched['Quantity'] * df_matched['UnitPrice_wms']
df_matched['Total_Value_variance'] = df_matched['Total_Value_wms'] - df_matched['Total_Value_oms']

print(f"Average UnitPrice variance: {df_matched['UnitPrice_variance'].mean():.4f}")
print(f"Max UnitPrice variance: {df_matched['UnitPrice_variance'].max():.4f}")
print(f"Min UnitPrice variance: {df_matched['UnitPrice_variance'].min():.4f}")


NUMERICAL VARIANCE
Average UnitPrice variance: -0.0074
Max UnitPrice variance: 0.0100
Min UnitPrice variance: -550.6400


Observation:
- The average UnitPrice variance of -0.0074 masks significant underlying pricing discrepancies, with variances ranging from -550.64 to +0.01. The extreme negative variance suggests missing prices in WMS (recorded as 0.00), while the minimal positive variance indicates only minor rounding differences. 
- After removing duplicates, the variance range normalized significantly, confirming that duplicates were artificially inflating extremes. 
- The asymmetric distribution with systematically lower WMS prices points to data quality issues rather than legitimate system differences. Despite the small average variance, the cumulative effect across 536,478 records creates measurable financial exposure requiring investigation

**Step 5** Duplicates and error impact

In [64]:
# Include flags from aggregated data
df_matched['has_flags_oms'] = df_matched['validation_flags_oms'].apply(lambda x: len(x) > 0 if isinstance(x, list) else False)
df_matched['has_flags_wms'] = df_matched['validation_flags_wms'].apply(lambda x: len(x) > 0 if isinstance(x, list) else False)

records_with_flags = df_matched[df_matched['has_flags_oms'] | df_matched['has_flags_wms']]
print(f"Matched records with validation flags: {len(records_with_flags)}")
print(f"Total variance from flagged records: {records_with_flags['Total_Value_variance'].sum():.2f}")

invalid_date_oms = df_matched['validation_flags_oms'].apply(lambda x: 'Invalid_Date' in x if isinstance(x, list) else False)
invalid_date_wms = df_matched['validation_flags_wms'].apply(lambda x: 'Invalid_Date' in x if isinstance(x, list) else False)
print(f"\nRecords with Invalid_Date flag in OMS: {invalid_date_oms.sum()}")
print(f"Records with Invalid_Date flag in WMS: {invalid_date_wms.sum()}")

duplicate_oms = df_matched['validation_flags_oms'].apply(lambda x: 'Duplicate_Record' in x if isinstance(x, list) else False)
duplicate_wms = df_matched['validation_flags_wms'].apply(lambda x: 'Duplicate_Record' in x if isinstance(x, list) else False)
print(f"Records with Duplicate_Record flag in OMS: {duplicate_oms.sum()}")
print(f"Records with Duplicate_Record flag in WMS: {duplicate_wms.sum()}")

missing_price_wms = df_matched['validation_flags_wms'].apply(lambda x: 'Missing_UnitPrice' in x if isinstance(x, list) else False)
print(f"Records with Missing_UnitPrice flag in WMS: {missing_price_wms.sum()}")

Matched records with validation flags: 6365
Total variance from flagged records: -17971.75

Records with Invalid_Date flag in OMS: 0
Records with Invalid_Date flag in WMS: 11
Records with Duplicate_Record flag in OMS: 4878
Records with Duplicate_Record flag in WMS: 5341
Records with Missing_UnitPrice flag in WMS: 992


Observation:
- Flagged records account for 6,365 matched transactions (1.19% of total), representing -17,971.75 in cumulative variance. 
- Invalid dates are isolated to WMS with 11 records, confirming that date validation issues do not affect OMS and have minimal financial impact. 
- Duplicate records are the dominant flag category with 4,878 in OMS and 5,341 in WMS, indicating systematic data entry or system synchronization problems. 
- Missing unit prices in WMS (992 records) represent the most significant data quality issue, directly contributing to negative price variances. 
- The concentration of flags in WMS suggests that the source system has more pronounced data quality challenges than OMS. 
- Flagged records account for approximately 89% of the total financial variance (-17,971.75 out of -17,884.93), demonstrating that identified data quality issues are the primary drivers of reconciliation discrepancies.

**Step 6** Financial impact

In [69]:
print("\n=== FINANCIAL IMPACT ===")

net_variance = df_matched['Total_Value_variance'].sum()
total_value_oms = df_matched['Total_Value_oms'].sum()
total_value_wms = df_matched['Total_Value_wms'].sum()
variance_pct = (net_variance / total_value_oms * 100) if total_value_oms != 0 else 0

print(f"Total OMS Value: {total_value_oms:,.2f}")
print(f"Total WMS Value: {total_value_wms:,.2f}")
print(f"Net Variance: {net_variance:,.2f}")
print(f"Variance %: {variance_pct:.2f}%")
print(f"Financial Exposure: {abs(net_variance):,.2f}")

# root Cause Cross-Referencing
print("\n=== ROOT CAUSE CROSS-REFERENCING ===")

discrepancy_report = df_matched[df_matched['UnitPrice_variance'] != 0][['InvoiceNo', 'StockCode', 'Quantity', 'UnitPrice_oms', 'UnitPrice_wms', 'UnitPrice_variance', 'Total_Value_variance']]

print(f"Records with discrepancies: {len(discrepancy_report)}")
print(discrepancy_report.head(10))


=== FINANCIAL IMPACT ===
Total OMS Value: 9,726,698.98
Total WMS Value: 9,708,814.05
Net Variance: -17,884.93
Variance %: -0.18%
Financial Exposure: 17,884.93

=== ROOT CAUSE CROSS-REFERENCING ===
Records with discrepancies: 1982
     InvoiceNo StockCode  Quantity  UnitPrice_oms  UnitPrice_wms  \
273     536396     21730         6           4.25           0.00   
771     536464     21815         1           1.45           1.46   
773     536464     21816         2           1.45           1.46   
824     536464    85231B         3           0.85           0.86   
1022    536522    47599B         1           2.10           2.11   
1092    536528     21992         1           2.95           0.00   
1556    536544     21935         1           3.36           0.00   
1868    536544     84988         1           2.98           0.00   
2198    536571     84754        12           1.25           0.00   
2502    536592     21656         1           3.36           0.00   

      UnitPrice_vari

Observation:
- The total OMS value of 9,726,698.98 compared to WMS value of 9,708,814.05 reveals a net variance of -17,884.93, representing a -0.18% discrepancy that translates to 17,884.93 in financial exposure. While the percentage variance appears minimal, the absolute financial exposure is material and requires resolution. 
- The analysis identified 1,982 records with price discrepancies, indicating that approximately 0.37% of all matched transactions have pricing mismatches. 
- The discrepancy patterns show two distinct categories: **minor variances** of +0.01 ((intentional price modifications introduced during data corruption phase) and **major negative variances** reaching -550.64 (intentional missing prices set to NaN during data corruption, recorded as 0.00 in WMS). 
- The concentration of discrepancies in specific invoices suggests systematic issues rather than random data corruption. 
- Missing unit prices account for the majority of negative variances, with multiple records showing OMS prices ranging from 1.25 to 4.25 while WMS records show 0.00. This pattern directly correlates with the Missing_UnitPrice validation flags identified in the Validation phase, confirming the traceability of data quality issues through the reconciliation pipeline.

In [ ]:
print("\n ADVANCED ROOT CAUSE ANALYSIS")

# Check the coverage between discrepancies and flags 
discrepancy_indices = df_matched[df_matched['UnitPrice_variance'] != 0].index

# Records with discrepancies and Missing_UnitPrice 
missing_price_in_discrepancies = df_matched.loc[discrepancy_indices, 'validation_flags_wms'].apply(
    lambda x: 'Missing_UnitPrice' in x if isinstance(x, list) else False
).sum()

print(f"Discrepancies with Missing_UnitPrice flag: {missing_price_in_discrepancies}")
print(f"Total discrepancies: {len(discrepancy_report)}")
print(f"Coverage: {(missing_price_in_discrepancies / len(discrepancy_report) * 100):.2f}%")

# Records with discrepancies and Duplicate_Record
duplicate_in_discrepancies_oms = df_matched.loc[discrepancy_indices, 'validation_flags_oms'].apply(
    lambda x: 'Duplicate_Record' in x if isinstance(x, list) else False
).sum()

duplicate_in_discrepancies_wms = df_matched.loc[discrepancy_indices, 'validation_flags_wms'].apply(
    lambda x: 'Duplicate_Record' in x if isinstance(x, list) else False
).sum()

print(f"\nDiscrepancies with Duplicate_Record flag in OMS: {duplicate_in_discrepancies_oms}")
print(f"Discrepancies with Duplicate_Record flag in WMS: {duplicate_in_discrepancies_wms}")

# Records with discrepancies and Invalid_Date
invalid_date_in_discrepancies = df_matched.loc[discrepancy_indices, 'validation_flags_wms'].apply(
    lambda x: 'Invalid_Date' in x if isinstance(x, list) else False
).sum()

print(f"Discrepancies with Invalid_Date flag: {invalid_date_in_discrepancies}")

# Records with discrepancies without any flag
no_flags_in_discrepancies = df_matched.loc[discrepancy_indices, 'has_flags_oms'] | df_matched.loc[discrepancy_indices, 'has_flags_wms']
no_flags_count = (~no_flags_in_discrepancies).sum()

print(f"\nDiscrepancies WITHOUT any validation flags: {no_flags_count}")
print(f"This suggests {no_flags_count} discrepancies are due to data quality issues not caught in Validation phase")



=== ADVANCED ROOT CAUSE ANALYSIS ===
Discrepancies with Missing_UnitPrice flag: 988
Total discrepancies: 1982
Coverage: 49.85%

Discrepancies with Duplicate_Record flag in OMS: 17
Discrepancies with Duplicate_Record flag in WMS: 0
Discrepancies with Invalid_Date flag: 0

Discrepancies WITHOUT any validation flags: 983
This suggests 983 discrepancies are due to data quality issues not caught in Validation phase


Observations:

- Missing_UnitPrice Coverage (49.85%): Nearly half of all discrepancies (988 out of 1,982) are directly linked to Missing_UnitPrice flags identified in the Validation phase. This demonstrates strong traceability between data quality issues and reconciliation failures. The remaining 994 discrepancies with missing prices were not flagged, suggesting either partial data corruption or system-specific pricing issues.

- Unflagged Discrepancies (983 records - 49.55%): Approximately half of all discrepancies have NO validation flags whatsoever. This is a critical finding indicating that the Validation phase missed significant data quality issues. These unflagged discrepancies likely represent:
    - Price mismatches due to system-specific rounding or calculation differences
    - Data entry errors that don't fit the predefined validation rules
    - System synchronization issues between OMS and WMS
    - Legitimate business logic differences (e.g., promotional pricing, regional adjustments)
- Duplicate_Record Impact (17 in OMS): Only 17 discrepancies are associated with Duplicate_Record flags in OMS, and none in WMS. This suggests that after aggregation, duplicates have minimal direct impact on price discrepancies, though they may have contributed to the overall data quality degradation.

- Invalid_Date Non-Impact (0 records): No discrepancies are linked to Invalid_Date flags, confirming that date issues do not directly cause price variances. This validates the separation of concerns in the Validation phase.

- Validation Phase Effectiveness: The Validation phase successfully identified ~50% of discrepancies but failed to catch the other 50%, indicating a need for enhanced validation rules to capture system-specific pricing anomalies and data entry errors.